# Week 9 — Wednesday: Missing Data

**DATA 202 · Calvin University**

Monday, the *Titanic* model learned from 1,043 passengers. We left out **266** because a blank sat in their age, fare or port.

> 🎯 **Who goes missing from a dataset — and what should a model do about the blanks?**

**Today's plan (~44 min):**

| Time | Part |
|---|---|
| ~6 min | 1 · Who is missing? |
| ~16 min | 2 · Three ways to go missing (SLO 09C) |
| ~4 min | 3 · Just drop them? |
| ~13 min | 4 · Fill them in, and which fix for which blank (SLO 09C) |
| ~5 min | 5 · Imputation inside the pipeline (SLO 09B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: *titanic3*, compiled by Thomas Cason from Encyclopedia Titanica; distributed by Frank Harrell, Vanderbilt Biostatistics</sub>

---
## 1 · Who Is Missing? · ~6 min

All **1,309** passengers this time.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

titanic = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/titanic.csv")
titanic.isna().sum()                                     # blanks in each column

`age`: 263 blanks · `cabin`: 1,014 · `embarked`: 2 · `fare`: 1. Monday's 266 = the 263 ages + 2 ports + 1 fare.

> 🎯 **Predict First:** Is a missing age equally likely in every class?

In [ ]:
titanic["no_age"] = titanic["age"].isna()

share = titanic.groupby("pclass")["no_age"].mean()
px.bar(share, title="Share of Passengers With No Recorded Age, by Class").show()

titanic.pivot_table(index="pclass", columns="no_age", values="survived")   # share who survived

> 📄 **Handout 1 — Who lost their age?** 3rd class lost the most ages, and in every class the passengers with no age survived less often. 🗣️ What story could explain both patterns?

<details><summary>Answer</summary>

* **3rd class:** 29% have no age. 1st: 12%. 2nd: 6%
* **Survival:** 41% with an age, 28% without. Even *inside* 1st class: 64% vs. 49%
* One possible story: ages were often pieced together afterwards, from documents and from survivors themselves. Passengers who died, and who traveled with fewer papers, left fewer traces. The data can't confirm it, but the blanks are clearly **not** random
</details>

---
## 2 · Three Ways to Go Missing (SLO 09C) · ~16 min

We can't see the real missing ages. So let's make blanks ourselves, where we know the answer: take the **1,046** passengers whose age *is* recorded, and imagine three clerks copying the register. Each one loses about 1 age in 5, in a different way:

| clerk | how the ages get lost |
|:-:|---|
| **1** | ink spills on random lines |
| **2** | copies from the tickets, and 3rd-class tickets often left the age off: loses 45% of 3rd-class ages, 5% of the others |
| **3** | asks the passengers, and the older a passenger, the likelier they refuse to say |

> 🎯 **Predict First:** For each clerk, will the lost ages be younger, older, or about the same as the ages kept?

In [ ]:
import numpy as np

known = titanic.dropna(subset=["age"]).copy()          # the passengers whose age we know
rng = np.random.default_rng(96)

# True = this clerk lost the age
known["clerk 1"] = rng.random(len(known)) < 0.20                                              # random lines
known["clerk 2"] = rng.random(len(known)) < known["pclass"].map({1: 0.05, 2: 0.05, 3: 0.45})  # depends on class
known["clerk 3"] = rng.random(len(known)) < ((known["age"] - 15) / 60).clip(0.02, 0.9)        # depends on the age itself

rows = []
for clerk in ["clerk 1", "clerk 2", "clerk 3"]:
    lost = known[clerk]
    rows.append({"": clerk, "ages lost": lost.sum(),
                 "mean age, lost": known.loc[lost, "age"].mean(),
                 "mean age, kept": known.loc[~lost, "age"].mean()})
pd.DataFrame(rows).set_index("").round(1)

<details><summary>Answer</summary>

* **Clerk 1:** lost 29.5, kept 30.0. The same: the kept ages are a smaller but fair sample
* **Clerk 2:** lost **25.8**, kept 31.2. Younger: 3rd class lost the most, and 3rd class was the youngest (mean 25, against 39 in 1st)
* **Clerk 3:** lost **41.4**, kept 25.7. Much older: the old refused
</details>

Clerks 2 and 3 both leave the kept ages wrong. Are they the same problem? Look **inside each class**:

In [ ]:
for clerk in ["clerk 2", "clerk 3"]:
    table = known.pivot_table(index="pclass", columns=clerk, values="age")       # mean age
    display(table.rename(columns={False: "kept", True: "lost"}).round(1).rename_axis(clerk, axis=1))

<details><summary>Answer</summary>

* **Clerk 2:** inside each class, lost ≈ kept (38.6 vs. 39.2 · 29.9 vs. 29.5 · 24.7 vs. 24.9). Once you know the class, a blank tells you nothing more about the age. **Class, a column we can see, explains the blanks**
* **Clerk 3:** inside every class, the lost ages are still about **13 years older** (48.6 vs. 33.3 · 38.6 vs. 25.9 · 35.4 vs. 22.3). Knowing the class doesn't help. **The blank depends on the age itself**
</details>

### The three mechanisms

**MCAR — missing completely at random** (clerk 1)
* What decides the blank: **nothing**, pure chance
* The rows left: a smaller but fair sample of everyone
* Cost: fewer rows, nothing worse

**MAR — missing at random** (clerk 2)
* What decides the blank: **other columns we can see** (here, `pclass`)
* The rows left: tilted overall (too few 3rd-class passengers, too old), but **fair inside each group** of those columns
* The name misleads: it is *not* random overall. Read it as "random **once you know** the other columns"
* So a fix can use those columns: fill a 3rd-class blank from 3rd-class passengers

**MNAR — missing not at random** (clerk 3)
* What decides the blank: **the missing value itself**, or something nobody recorded
* The rows left: tilted overall **and inside every group**. No column we have can correct it
* No fill brings the true values back. At most, the blank itself becomes a clue

**MAR or MNAR? One question:** take passengers who are alike in every column we *can* see. Among them, do the blank ones still differ?
* **No → MAR.** **Yes → MNAR**
* We could check it today only because we made the blanks and kept the true ages. With real blanks, the values we'd need are exactly the missing ones. The data can rule out MCAR (blanks that differ by group); MAR vs. MNAR has to come from knowing **how the data was collected**

In one table:

| | the blank depends on… | example |
|---|---|---|
| **MCAR** — missing completely at random | nothing: pure chance | a clerk spills ink on one line |
| **MAR** — missing at random | other columns we **can** see | more blank ages in 3rd class |
| **MNAR** — missing not at random | the **missing value itself**, or something we never recorded | people hide their age *because* of their age |

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/missing_types.png" width="520" alt="Three panels. MCAR: a grid with red missing cells scattered at random. MAR: a column with missing cells, whose missingness can be explained using the observed data. MNAR: a column whose missingness is linked to the missing values themselves, or to unobserved features.">

<sub>Image: Daily Dose of Data Science (blog.DailyDoseofDS.com)</sub>

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/09/images/manuscript.jpeg" width="520" alt="A page of an old handwritten manuscript in Spanish, with stains and faded words.">

> 💬 **Question:** On an old manuscript, water stains erased words wherever the drops fell. A later owner scraped out the passages he disagreed with. Which blanks are MCAR, and which are MNAR?

<details><summary>Answer</summary>

* **Water stains → MCAR:** where a drop fell has nothing to do with what was written there
* **Scraping → MNAR:** the content of the missing words decided which words went missing. Reading only what's left, you would never know what the owner disliked
</details>

> 📄 **Handout 2 — Name the mechanism.** Write MCAR, MAR or MNAR for each situation. 🗣️ Compare with your neighbor, then defend your answers to 10 and 11.

1. A lab freezer breaks overnight and spoils a random rack of blood samples
2. To keep an online survey short, a randomly chosen half of the people never see the income question
3. Two *Titanic* passengers have no port: a clerk smudged their line in the register
4. A weather station's thermometer stops working below −30 °C
5. In a salary survey, the people with the highest salaries skip the salary question
6. A fitness watch records no steps on the days its owner leaves it at home: mostly lazy days
7. A clinic measures cholesterol only for patients over 40. Every patient's age is in the data
8. Men skip the "hours of sleep" question more often than women, however much they sleep. Sex is in the data
9. Cabin is blank for 98% of *Titanic* 3rd class: their tickets rarely listed one
10. Students who were failing a course dropped it, so their final grade is blank. Their midterm grades are in the data
11. The real missing *Titanic* ages from Part 1

<details><summary>Answer</summary>

| | mechanism | why |
|---|---|---|
| 1 | **MCAR** | which rack spoiled has nothing to do with the patients |
| 2 | **MCAR** | chance decided, on purpose: "planned missingness" |
| 3 | **MCAR** | a smudge has nothing to do with the passenger |
| 4 | **MNAR** | the blank depends on the temperature itself: the coldest readings vanish |
| 5 | **MNAR** | the blank depends on the salary itself |
| 6 | **MNAR** | the missing step counts would have been low ones |
| 7 | **MAR** | age, which we can see, decides who has a blank |
| 8 | **MAR** | sex, which we can see, explains the blanks |
| 9 | **MAR** | `pclass` explains it. (A column we'd drop anyway: 77% blank) |
| 10 | **MAR, mostly** | the midterm shows who was failing. Without the midterm column, the same blanks would be **MNAR** |
| 11 | **not MCAR; probably not only MAR** | class explains part (29% vs. 6%), but even *inside* a class the blanks go with dying, and with papers no one recorded: MNAR |

* The label depends on **what else we recorded** (10): a new column can turn MNAR into MAR
* The data can rule out MCAR. It can never prove MAR vs. MNAR, because the evidence is exactly what's missing
</details>

The mechanism matters because each fix quietly assumes one of them.

---
## 3 · Just Drop Them? · ~4 min

Monday's choice: keep only the complete rows.

In [ ]:
complete = titanic.dropna(subset=["age", "fare", "embarked"])

print("passengers:", len(titanic), "→", len(complete))
print("survived:  ", round(titanic["survived"].mean(), 3), "→", round(complete["survived"].mean(), 3))
print("3rd class: ", round((titanic["pclass"] == 3).mean(), 3), "→", round((complete["pclass"] == 3).mean(), 3))

<details><summary>Answer</summary>

* 3rd class: **54% → 48%** of the passengers. Survival: **38.2% → 40.7%**
* Dropping rows is safe only under **MCAR**. Here the model learns from a richer, luckier ship than the real one
</details>

And on the day a new passenger arrives with no age?

In [ ]:
X = titanic[["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
y = titanic["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)

print(len(X_train), "training |", len(X_test), "test")
print("test passengers with a blank:", X_test.isna().any(axis=1).sum())

A model trained on complete rows can't answer for those **75** test passengers: kNN can't measure a distance to a blank.

---
## 4 · Fill Them In: Imputation (SLO 09C) · ~13 min

**Imputation:** replace each blank with a guess, learned from the **training** rows.

The simplest guess is fix **B**: one value for everyone, the column's mean (or median). Every passenger with no age gets the age of the "average" passenger on the ship.

> 📄 **Handout 3 — One age for everyone.** Each panel is one class: recorded ages, and the blanks filled with the ship's mean age. Draw an arrow to where you'd put each class's blanks instead. 🗣️ One mean assumes the blank passengers are like everyone else. Is that true here? Which mechanism from Part 2 does it assume, and which one do our blanks follow?

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputer.fit(X_train[["age"]])                    # learns the training passengers' mean age
print("mean age:", imputer.statistics_)

ages = X_train[["pclass"]].copy()
ages["age"] = imputer.transform(X_train[["age"]])[:, 0]
ages["age is"] = X_train["age"].isna().map({False: "recorded", True: "filled with the mean"})
print("standard deviation:", X_train["age"].std(), "→", ages["age"].std())

px.histogram(ages, x="age", color="age is", facet_row="pclass", nbins=40, height=600,
             title="Training Passengers' Ages by Class, Blanks Filled With One Mean").show()

<details><summary>Answer</summary>

* All **189** training passengers with no age are now exactly 30.0: a spike no real ship had. The spread shrinks (standard deviation **14.7 → 13.2** years), so the data looks more certain than it is
* **146** of the 189 are in 3rd class, where the median age is **24**: the mean of 30 makes them too old. The 30 blank 1st-class passengers get 30 too, against a median of **39**: too young
* One mean assumes the blank passengers are like everyone else: **MCAR**. Our blanks are **MAR** (class explains who is blank, Part 1), so the fill is wrong in a predictable direction for each class
* Better: guess from the passenger's own class, or from passengers like them (fix **C**). That's the next table
</details>

Better guesses use the columns we *can* see:

| | fills a blank with… | `sklearn.impute` |
|---|---|---|
| mean / median / most frequent | one value for everyone | `SimpleImputer` |
| similar passengers | the average of the k most similar rows | `KNNImputer` (scale first: Monday!) |
| a model's prediction | a regression from the other columns, repeated | `IterativeImputer` |

Or keep the blank as information: `add_indicator=True` fills the blank **and** adds a second column that is 1 when the value was missing. Eight test passengers:

In [ ]:
imputer = SimpleImputer(strategy="median", add_indicator=True)
imputer.fit(X_train[["age"]])

eight = X_test[["age"]].head(8)
display(eight)
imputer.transform(eight)                          # column 1: age, blanks filled · column 2: was it blank?

> "The simplest eigenstatus of all is an indicator of mere presence or absence, observed or unobserved, on some dimension. If you are not included, you cannot be measured or assessed."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** With the indicator, the model can learn "no age recorded → less likely to survive." Is that a fact about the passengers, or about the people who kept the records? Should the model use it?

<details><summary>Answer</summary>

* It's both. The blank traces back to who left papers, who was asked afterwards, and who could still answer
* For predicting it may help: the blank carries real information when it's MNAR
* But the model then scores people partly on whether an institution recorded them. Today that might be a thin credit file or a missing medical history
</details>

### Which fix for which blank?

| | fix | what it assumes about the blank rows |
|:-:|---|---|
| **A** | drop the rows with blanks | they are like everyone else |
| **B** | fill with one mean or median | they are like everyone else |
| **C** | fill from similar rows (`KNNImputer`) | other columns explain who is blank |
| **D** | C, plus a missing-indicator column | the blank itself says something |

> 📄 **Handout 4 — Pick the fix.** Five situations from Handout 2. Write the best letter for each. 🗣️ For the salaries: does any letter bring the missing salaries back?

1. The freezer spoiled a random rack of blood samples *(#1)*
2. Men skip the sleep question; sex is in the data *(#8)*
3. Failing students dropped the course; their midterms are in the data *(#10)*
4. The highest earners skip the salary question *(#5)*
5. The real missing *Titanic* ages *(#11)*

<details><summary>Answer</summary>

1. **A or B** (MCAR): the blank rows are a random sample, so dropping them biases nothing. **A** if the blanks are few; **B** if dropping would lose many rows, or new samples will arrive with blanks
2. **C** (MAR): fill a man's sleep from other men like him. One overall mean would mix in the women's answers
3. **C** (MAR): the midterm predicts the final grade. Dropping these students, or giving them the class mean, makes the course look easier than it was
4. **D** (MNAR), and it is still not fixed. No letter brings the salaries back: the indicator only lets the model use "didn't say" as a clue. Beyond the model: ask why people skip, offer salary ranges, and report that top earners are underrepresented
5. **D** (MAR + maybe MNAR): class explains part of the blanks (C), and the rest may depend on what we never recorded (the indicator). That's the pipeline we build next
</details>

---
## 5 · Imputation Inside the Pipeline (SLO 09B) · ~5 min

Our ages are MAR, and maybe MNAR: fix **D**. Monday's pipeline, with an imputer added to each branch. The imputer learns from the training rows only, like the scaler.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ["pclass", "age", "sibsp", "parch", "fare"]
categorical = ["sex", "embarked"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5, add_indicator=True)), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(drop="if_binary")), categorical),
])          # make_pipeline: run these steps one after the other. Scale first: KNNImputer measures distances
model = Pipeline([
    ("prep", preprocess),
    ("knn", KNeighborsClassifier(n_neighbors=25)),
])
model

> 🎯 **Predict First:** Fixes A and B assume the wrong mechanism for our ages. Will they score much worse than D?

In [ ]:
print("CV accuracy:", cross_val_score(model, X_train, y_train, cv=5).mean())

model.fit(X_train, y_train)
no_age = X_test["age"].isna()
print("test accuracy:", model.score(X_test, y_test))
print("test passengers with no age:", model.score(X_test[no_age], y_test[no_age]))
print("lazy model, 'everyone died':", (y_test == 0).mean())

<details><summary>Answer</summary>

No. The same pipeline, swapping only the imputer (5-fold CV on the training passengers):

| fix | strategy | training rows | CV accuracy |
|:-:|---|---|---|
| A | drop incomplete rows | 790 | 0.781 |
| B | mean | 981 | 0.785 |
| B | median | 981 | 0.785 |
| B + indicator | median + indicator | 981 | 0.787 |
| C | 5 similar passengers | 981 | 0.786 |
| **D** | **5 similar passengers + indicator** (ours) | 981 | **0.785** |

* The 5 folds disagree by up to 10 points, so these differences are noise: **the score can't tell the mechanisms apart**
* What the fixes *do* differ on:
  * **who is included:** A loses 191 training passengers, mostly 3rd class, and can't answer for 75 test passengers
  * **what they assume:** B treats the blanks as MCAR; C uses class and the other columns (MAR); D also treats the blank itself as a signal (MNAR)
* Choose by the mechanism you believe, then check the score. Not the other way around
</details>

> 📌 **One split:** test accuracy 0.765, and 0.730 on the 74 test passengers with no age. Every passenger got an answer.

---
### Wrap-up: which fix for which blank?

| the blanks are… | example | best fix | in sklearn | what goes wrong otherwise |
|---|---|---|---|---|
| **MCAR**: pure chance | a freezer spoils a random rack of samples | **A** drop the rows (if few), or **B** fill with the mean or median | `dropna()`, `SimpleImputer` | little: you lose rows (A) or shrink the spread (B) |
| **MAR**: explained by columns we see | men skip the sleep question | **C** fill from similar rows, using the columns that explain the blank | `KNNImputer` (scale first), `IterativeImputer` | A and B tilt the data toward the groups with fewer blanks |
| **MNAR**: depends on the hidden value | top earners skip the salary question | **D** fill + missing indicator; find a column that explains the blank; say the limit out loud | `add_indicator=True` | every fill looks complete but is wrong for exactly the blank rows |

* **Can't tell MAR from MNAR?** Usually you can't. Plan for both with **D**, as we did for the *Titanic*
* **Look before you fill:** count the blanks, and ask *who* they belong to
* **Imputation goes inside the pipeline**, learned from training rows only 📌